# 10 · Genomic Filtering (R)

This notebook teaches the genomic-filtering API in the R adapter. By the end you'll be able to filter a query by genetic **variant annotations** (gene, variant consequence, population frequency) and pull back either **patient** counts or **variant**-level results. It mirrors the Python adapter's `10_Genomic_Filtering` notebook.

### The mental model

A PIC-SURE query filters along two independent axes that combine with AND:

1. **Phenotypic** — clinical/observational traits (age, sex, a diagnosis). These are the clauses from `picsure::buildClause()` / `picsure::buildClauseGroup()`.
2. **Genomic** — properties of the genetic *variants* a participant carries. These come from `picsure::buildGenomicFilter()`.

Unlike the phenotypic side (an AND/OR *tree*), genomic filters are a **flat list applied conjunctively** — every filter in the list must match. A query can use either axis, or both.

> Many cells below (building filters, inspecting `$to_query_json()`, the value enum) don't hit a server — but the first call still provisions the reticulate Python env. The cells that call `runQuery()` need a connection to a deployment that has genomic data.

## Setup

Genomic data lives on **authorized** BDC, so connect with a token. Put your PIC-SURE token in `notebooks/token.txt` (one line).

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

### Configure the examples

Genomic annotation **keys** and **values** are deployment-specific. The keys below (`Gene_with_variant`, `Variant_consequence_calculated`, `Variant_frequency_as_text`) are the standard BDC annotations; adjust the gene symbol to match your data. You can discover valid genes and consequences from the adapter (next section) or in the PIC-SURE UI's variant explorer.

In [ ]:
# --- edit these to match your deployment / cohort of interest ---
GENE <- "CHD8"            # a gene symbol present in the variant data
SEARCH_TERM <- "sex"      # any phenotypic concept you can filter on

# Standard BDC genomic annotation keys (usually stable):
GENE_KEY <- "Gene_with_variant"
CONSEQUENCE_KEY <- "Variant_consequence_calculated"
FREQ_BUCKET_KEY <- "Variant_frequency_as_text"

### Discovering valid genes & consequences

You don't have to hardcode genes and consequence values. On an **authorized** platform, `picsure::searchGenomicValues(session, key, query = ...)` looks them up from the server — paginated and searchable like the UI's autocomplete (raise `size` to pull more per call). It works for any genomic key. For consequences there's also `picsure::genomicConsequences()`, the built-in vocabulary grouped by severity, which needs no connection.

In [ ]:
# Search genes with variants by prefix (authorized platforms only)
picsure::searchGenomicValues(session, GENE_KEY, query = "BRCA", size = 20)

In [ ]:
# searchGenomicValues works for any genomic key -- e.g. consequence values:
picsure::searchGenomicValues(session, CONSEQUENCE_KEY, query = "splice")

In [ ]:
# The built-in consequence vocabulary, grouped by severity (offline, no server):
picsure::genomicConsequences()

## 1. Anatomy of a genomic filter

`picsure::buildGenomicFilter(key, values = ...)` builds one filter. A filter is **categorical**: you name the annotation (`key`) and the `values` that count as a match. Build a gene filter and inspect the exact JSON it sends over the wire with `$to_query_json()` (no server needed):

In [ ]:
gene_filter <- picsure::buildGenomicFilter(GENE_KEY, values = c(GENE))
gene_filter$to_query_json()

`values` accepts a single string or a vector — pass several values to match any of them. For example, a few variant-consequence categories at once:

In [ ]:
consequences <- picsure::buildGenomicFilter(
  CONSEQUENCE_KEY,
  values = c("missense_variant", "stop_gained")
)
consequences$to_query_json()

### Value enum

A small fixed vocabulary ships as an enum for discoverability: `VariantFrequency` (Rare/Common/Novel buckets for the `Variant_frequency_as_text` key). Pass a member **directly** to `values =` — the wrapper coerces it to its string value.

In [ ]:
sapply(picsure::VariantFrequency, function(m) m$value)

In [ ]:
rare_bucket <- picsure::buildGenomicFilter(
  FREQ_BUCKET_KEY,
  values = picsure::VariantFrequency$RARE
)
rare_bucket$to_query_json()

## 2. A genomic filter as a constraint on a patient count

Attach a genomic filter to a query and run it like any other. With the patient-centric result types (`COUNT`, `PARTICIPANT`), the genomic filter acts as a **constraint**: *participants who carry at least one matching variant*. A query may be **genomic-only** — no phenotypic filter required.

In [ ]:
gene_query <- picsure::buildQuery(genomicFilters = gene_filter)

count <- picsure::runQuery(session, gene_query, type = picsure::QueryType$COUNT)
count$value

`count` is a `CountResult`. On small cohorts the server obfuscates the number, so check the slots rather than assuming `$value` is set:

In [ ]:
if (!is.null(count$value)) {
  cat(count$value, "participants carry a qualifying", GENE, "variant\n")
} else {
  cat("fewer than", count$cap, "participants (suppressed)\n")
}

## 3. Combining genomic + phenotypic filters

Pass both `phenotypicFilter =` and `genomicFilters =` to `buildQuery()`. They combine with AND: *matches the phenotype **and** carries a matching variant.* Here we find any phenotypic concept by search and use a `REQUIRE` clause (matches participants who have a value for it):

In [ ]:
results <- picsure::searchDictionary(session, SEARCH_TERM)
concept_path <- results$conceptPath[[1]]
concept_path

In [ ]:
phenotypic <- picsure::buildClause(
  concept_path,
  type = picsure::PhenotypicFilterType$REQUIRE
)

combined <- picsure::buildQuery(
  phenotypicFilter = phenotypic,
  genomicFilters   = gene_filter
)

picsure::runQuery(session, combined, type = picsure::QueryType$COUNT)$value

## 4. Multiple genomic filters (ANDed)

Pass a **list** to narrow the variant set further — e.g. *rare, loss-of-function variants in our gene*. Each entry must match:

In [ ]:
filters <- list(
  picsure::buildGenomicFilter(GENE_KEY, values = c(GENE)),
  picsure::buildGenomicFilter(
    CONSEQUENCE_KEY,
    values = c("stop_gained", "frameshift_variant")
  ),
  picsure::buildGenomicFilter(FREQ_BUCKET_KEY, values = picsure::VariantFrequency$RARE)
)

multi <- picsure::buildQuery(genomicFilters = filters)
lapply(filters, function(f) f$to_query_json())

In [ ]:
picsure::runQuery(session, multi, type = picsure::QueryType$COUNT)$value

## 5. Variant-centric result types

The result types above answer questions about **people**. Four result types answer questions about the **variants** themselves:

| `type =`                         | returns            | question |
|----------------------------------|--------------------|----------|
| `QueryType$VARIANT_COUNT`         | `CountResult`      | how many distinct variants match? |
| `QueryType$VARIANT_LIST`          | character vector   | which variants match? |
| `QueryType$VCF_EXCERPT`           | data.frame         | a VCF-style table, **with** per-patient genotype columns |
| `QueryType$AGGREGATE_VCF_EXCERPT` | data.frame         | same table, **without** patient columns (privacy-preserving) |

All four respect the same genomic + phenotypic filters.

> **Deployment note (as of 2026-06):** BDC's hosted PIC-SURE does not serve these variant result types yet — it returns an empty response or an HTTP 500, which the adapter surfaces as a clear `picsureError` ("...not available on this PIC-SURE deployment yet"). Genomic filtering as a **constraint** (Sections 2–4, using `COUNT` / `PARTICIPANT`) works today. The cells below will raise that error on BDC until the backend enables variant output; they're shown so you know the API.

In [ ]:
# How many distinct variants match? (a CountResult, like a patient count)
variant_count <- picsure::runQuery(session, multi, type = picsure::QueryType$VARIANT_COUNT)
variant_count$value

`VARIANT_LIST` returns the matching variant specs as a character vector. Each spec is six comma-separated fields: `chromosome,offset,ref,alt,gene,consequence`.

In [ ]:
variants <- picsure::runQuery(session, gene_query, type = picsure::QueryType$VARIANT_LIST)
cat(length(variants), "variants\n")
head(variants, 5)

In [ ]:
# VCF excerpt: one row per variant, with per-patient genotype columns.
vcf <- picsure::runQuery(session, gene_query, type = picsure::QueryType$VCF_EXCERPT)
head(vcf)

In [ ]:
# Aggregate VCF excerpt: same rows, without the per-patient columns.
agg <- picsure::runQuery(session, gene_query, type = picsure::QueryType$AGGREGATE_VCF_EXCERPT)
head(agg)

## 6. Saving & loading genomic queries

Genomic filters survive a round-trip. On an authorized platform you can `saveQueryByName()` a query that carries genomic filters and later `loadQueryByID()` it back into an equivalent query handle — including gene/consequence/frequency queries originally built in the PIC-SURE UI's variant explorer.

```r
qid <- picsure::saveQueryByName(session, multi, "Rare LoF in my gene")
restored <- picsure::loadQueryByID(session, qid)
restored$genomicFilters
```

## Recap / cheat sheet

- **Discover values:** `picsure::searchGenomicValues(session, key, query = ...)` (authorized; genes, consequences, any genomic key) and `picsure::genomicConsequences()` (offline consequence vocabulary by severity).
- **Build one filter:** `picsure::buildGenomicFilter(key, values = ...)` — one value or a vector; matches any of them.
- **Attach to a query:** `picsure::buildQuery(genomicFilters = one_or_a_list)`; combine with `phenotypicFilter =` to AND the two axes. Genomic-only queries are fine.
- **Patient results** (`COUNT`, `PARTICIPANT`): the genomic filter is a constraint on *who* matches.
- **Variant results** (`VARIANT_COUNT`, `VARIANT_LIST`, `VCF_EXCERPT`, `AGGREGATE_VCF_EXCERPT`): answers about the *variants*.
- **Enum:** `VariantFrequency` (Rare/Common/Novel) — pass members straight to `values =`.
- **Gotchas:** `VARIANT_COUNT` is a `CountResult` (handles obfuscation), not a bare integer; `VARIANT_LIST` specs are comma-delimited `chrom,offset,ref,alt,gene,consequence`; keys/genes are deployment-specific — discover them with `searchGenomicValues()` or the PIC-SURE UI. Variant-spec (SNP) filtering is not supported yet.